In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_parquet('renyi_data.parquet')
data = df.pivot_table(index='date', columns='csecid', values='returns').values

In [3]:
def sample(data, size):
    o = np.empty((size, data.shape[1]))
    r = np.random.default_rng(42)
    for it in range(size):
        ii = r.integers(0, data.shape[0])
        o[it] = data[ii]
    return o

In [4]:
rng = np.random.default_rng(42)
p = 0.125


idx = rng.integers(0, data.shape[0])
out = np.empty(data.shape)

for t in range(data.shape[0]):
    out[t] = data[idx]
    if rng.random() < p:
        idx = rng.integers(0, data.shape[0])
    else:
        idx = (idx + 1) % data.shape[0]

In [5]:
N = 2048
final = sample(out, N)
np.save('SB2048.npy', final.T)

In [6]:
a = np.load("PSM2048.npy")
b = np.load("ROBECO.npy")
c = np.load("PSMDiff2048.npy")
d = np.load("SB2048.npy")
e = np.load("PSMResample2048.npy")
a.shape, b.shape, c.shape, d.shape, e.shape

((2048, 3033), (3914, 3033), (2048, 3033), (3033, 2048), (2048, 3033))

In [7]:

# Generate temporal samples [M, N, D] = [2048, 81, 22]
def sample_temporal(data, M, D, p=0.125, seed=42):
    """
    Stationary bootstrap for temporal sequences.
    Returns shape [M, N, D]: M scenarios, N assets, D days.
    Each scenario is a D-day path generated by the stationary bootstrap Markov chain.
    """
    T, N = data.shape
    rng = np.random.default_rng(seed)
    out = np.empty((M, N, D))
    for m in range(M):
        idx = rng.integers(0, T)
        for d in range(D):
            out[m, :, d] = data[idx]
            if rng.random() < p:
                idx = rng.integers(0, T)
            else:
                idx = (idx + 1) % T
    return out
